# Fase 1 — Esplorazione dati + Defense A (regex)

Progetto **The Guardrail Comparison**. Questo notebook copre EDA, split canonico, costruzione e congelamento del filtro a regex.

> Runtime su **CPU** (niente GPU in questa fase). Esegui prima `00_setup.ipynb`, oppure usa la cella di bootstrap qui sotto. Adatta `TUO_UTENTE` e i path di Drive.

## Bootstrap Colab

In [ ]:
# Esegui dopo 00_setup.ipynb, oppure standalone
import os
from google.colab import drive
drive.mount('/content/drive')

REPO = "/content/guardrail-comparison"
if not os.path.isdir(REPO):
    !git clone https://github.com/TUO_UTENTE/guardrail-comparison.git {REPO}
%cd {REPO}
!pip install -q -r requirements.txt

## Caricamento dati + integrity check

Usa il loader di `common.py`, che normalizza i nomi delle colonne del CSV dell'articolo e deriva `ground_truth` dalla colonna `Human` (1 = unsafe, 0 = safe).

In [ ]:
try:
    from common import set_seed, file_hash, load_dataset_csv, benchmark_latency
except ModuleNotFoundError:
    raise SystemExit("common.py non trovato: assicurati di essere nella root della repo (vedi Fase 0).")

import pandas as pd
set_seed(42)

CSV_PATH = "/content/drive/MyDrive/guardrail-data/dataset.csv"  # adatta il path
print("hash sha256:", file_hash(CSV_PATH))

df = load_dataset_csv(CSV_PATH)
print("righe:", len(df))
print("colonne normalizzate:", list(df.columns))
print(df["ground_truth"].value_counts())
df[["response_text", "category", "source", "ground_truth"]].head()

## Step 1.1 — EDA mirata

Bilanciamento delle etichette, struttura per `source`/`category`, e lunghezza delle risposte (utile per `max_seq_len` in Fase 2).

In [ ]:
print(df["ground_truth"].value_counts(normalize=True))   # bilanciamento safe/unsafe
print(df["source"].value_counts())                       # base vs xbreaking
print(df["category"].value_counts())                     # categorie di rischio

# dove si concentra l'unsafe?
print(pd.crosstab(df["category"], df["ground_truth"]))
print(pd.crosstab(df["source"], df["ground_truth"]))

# lunghezza delle risposte
df["len_char"] = df["response_text"].str.len()
print(df["len_char"].describe(percentiles=[.5, .9, .95, .99]))

## Step 1.2 — Controllo duplicati e leakage

Se la stessa risposta finisce sia nel train sia nel test, entrambi i filtri \"barano\". Deduplica **prima** di tutto, su testo normalizzato.

In [ ]:
df["norm"] = df["response_text"].str.lower().str.strip()
print("duplicati esatti (normalizzati):", df["norm"].duplicated().sum())
df = df.drop_duplicates(subset="norm").reset_index(drop=True)
print("righe dopo dedup:", len(df))

## Step 1.3 — Split canonico (una volta sola, condiviso)

Un solo split train/test, stratificato per categoria, salvato su Drive e riusato sia dal mining dei termini regex (qui) sia dal training del LLM (Fase 2). Da adesso il **test set e' intoccabile**.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["category"]
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

OUT_DIR = "/content/drive/MyDrive/guardrail-data"
train_df.to_csv(f"{OUT_DIR}/train.csv", index=False)
test_df.to_csv(f"{OUT_DIR}/test.csv", index=False)
print("train:", len(train_df), "| test:", len(test_df))

## Step 1.4 — Mining dei termini (solo dal train)

Non i termini piu' frequenti in assoluto, ma quelli che compaiono **molto piu' nelle unsafe che nelle safe** (frequenza differenziale).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

unsafe_txt = train_df[train_df["ground_truth"] == "unsafe"]["response_text"]
safe_txt   = train_df[train_df["ground_truth"] == "safe"]["response_text"]

vec = CountVectorizer(ngram_range=(1, 2), min_df=5, stop_words="english")
X_un = vec.fit_transform(unsafe_txt)
vocab = np.array(vec.get_feature_names_out())

freq_un = np.asarray(X_un.sum(axis=0)).ravel() + 1     # smoothing
freq_sa = np.asarray(vec.transform(safe_txt).sum(axis=0)).ravel() + 1

ratio = freq_un / freq_sa
order = np.argsort(ratio)[::-1]

candidati = [(vocab[i], int(freq_un[i] - 1), round(float(ratio[i]), 1))
             for i in order if (freq_un[i] - 1) >= 5][:100]
for term, n, r in candidati[:50]:
    print(f"{term:30s}  occorrenze={n:4d}  ratio={r}")

## Step 1.5 — Curare la blocklist a mano

I candidati automatici vanno **rivisti a mano**: i termini ambigui generano i falsi positivi del regex (over-blocking), uno dei risultati centrali. La cella sotto crea uno **starter** `defense_regex/blocklist.py` solo se non esiste gia' (cosi' non sovrascrive la tua curatura). Dopo la modifica a mano, committa il file.

In [ ]:
import os
os.makedirs("defense_regex", exist_ok=True)
open("defense_regex/__init__.py", "a").close()

STARTER = r'''# defense_regex/blocklist.py  (versionata e poi CONGELATA)
# Punto di partenza: aggiungi i termini curati a mano dallo Step 1.4.
BLOCKLIST = {
    "explicit_instructions": [
        r"\bstep[-\s]?by[-\s]?step\b",
        r"\bhere(?:'s| is) how to\b",
    ],
    "dangerous_domain": [
        # termini di dominio chiaramente unsafe ricavati dal mining
    ],
    "evasion_markers": [
        r"\bfor educational purposes only\b",
        r"\bhypothetically\b",
    ],
}
'''

path = "defense_regex/blocklist.py"
if os.path.exists(path):
    print("esiste gia', NON sovrascrivo (modificalo a mano e committalo):", path)
else:
    with open(path, "w") as f:
        f.write(STARTER)
    print("creato starter:", path)

## Step 1.6 — Implementare il filtro

`RegexDefense` rispetta il contratto di `common.py`. La cella `%%writefile` scrive il file nella repo (e' codice stabile, ok sovrascrivere).

In [ ]:
%%writefile defense_regex/regex_filter.py
import re, time
from common import Defense, ClassificationResult
from defense_regex.blocklist import BLOCKLIST


class RegexDefense(Defense):
    name = "regex"

    def __init__(self, normalize: bool = True):
        self.normalize = normalize
        self.compiled = {
            cat: [re.compile(p, re.IGNORECASE) for p in pats]
            for cat, pats in BLOCKLIST.items()
        }

    def _norm(self, text: str) -> str:
        if not self.normalize:
            return text
        text = text.lower()
        text = re.sub(r"\s+", " ", text)          # collassa spazi
        text = text.replace("0", "o").replace("1", "i").replace("3", "e")
        return text

    def classify(self, response_text: str) -> ClassificationResult:
        start = time.perf_counter()
        text = self._norm(response_text)
        hit = None
        for cat, regexes in self.compiled.items():
            for rx in regexes:
                if rx.search(text):
                    hit = f"{cat}:{rx.pattern}"
                    break
            if hit:
                break
        latency_ms = (time.perf_counter() - start) * 1000
        verdict = "unsafe" if hit else "safe"
        return ClassificationResult(
            verdict=verdict,
            score=1.0 if hit else 0.0,   # il regex e' binario: e' parte della storia
            latency_ms=latency_ms,
            matched_rule=hit,
        )

In [ ]:
# carica (o ricarica) il filtro appena scritto
import importlib, defense_regex.regex_filter as rf
importlib.reload(rf)
from defense_regex.regex_filter import RegexDefense

regex = RegexDefense()
print("filtro pronto:", regex.name)

## Step 1.7 — Misurare la latenza

Stesso metodo che userai per il LLM in Fase 2, cosi' il confronto e' onesto.

In [ ]:
campione = test_df["response_text"].tolist()[:300]
stats = benchmark_latency(lambda x: regex.classify(x), campione, repeats=10)
print(stats)  # median_ms piccolissimo -> riportalo in microsecondi

## Step 1.8 — Sanity check e congelamento

Il regex gira sul test del CSV solo come sanity check, **non** per ottimizzarlo.

In [ ]:
test_df["regex_pred"] = test_df["response_text"].apply(lambda x: regex.classify(x).verdict)
print(pd.crosstab(test_df["ground_truth"], test_df["regex_pred"]))

Congela con un tag git (da terminale, dopo aver curato la blocklist):

```bash
git add defense_regex/ notebooks/01_eda_regex.ipynb
git commit -m "Fase 1: EDA, split canonico, blocklist curata, RegexDefense"
git tag fase1-regex-frozen
git push --tags
```

**Prossimo passo:** Fase 2 — training del Guard LLM sullo stesso split canonico.